In [26]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from sklearn.preprocessing import PowerTransformer

In [27]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [28]:
# Need to choose patient_id from OUS_D3 in response_OUS
data = list(OUS_D3['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D3 with response_OUS
clinical_train = pd.merge(OUS_D3, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS', 'LRC', 'event_LRC'])]

In [29]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [30]:
# Check null values in D3 
clinical_train.isnull().sum().sum()

0

### COX assumption in Train data

In [31]:
df = pd.DataFrame(clinical_train)

# Fit Cox proportional hazards model
cph = CoxPHFitter(penalizer=0.00001)
cph.fit(df, duration_col='DFS', event_col='event_DFS')

# Perform the proportional hazards assumption test
results = proportional_hazard_test(cph, df)
print(results)

<lifelines.CoxPHFitter: fitted with 139 total observations, 71 right-censored observations>

<lifelines.StatisticalResult: proportional_hazard_test>
    time_transform = rank
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 139 total observations, 71 right-censored observations>
         test_name = proportional_hazard_test

---
                                                       test_statistic    p  -log2(p)
LBP_003_CT                                                       0.01 0.91      0.13
LBP_003_PET                                                      0.00 0.98      0.03
LBP_012_CT                                                       0.00 0.95      0.08
LBP_012_PET                                                      0.00 0.97      0.04
LBP_021_CT                                                       0.00 0.97      0.05
LBP_021_PET                                                      0.01 0.92      0.12
LBP_030_CT                                                       0.00 0.99      0.02
LBP_030_PET       

In [32]:
# Access the summary table
summary_table = results.summary

# Filter columns based on p-values lower than 0.05
significant_columns = summary_table[summary_table['p'] < 0.05].index

# Display significant columns
print("Columns with p-values < 0.05:")
print(significant_columns)

Columns with p-values < 0.05:
Index([], dtype='object')


In [33]:
df = pd.DataFrame(clinical_train)

# Fit Cox proportional hazards model
cph = CoxPHFitter(penalizer=0.1)
cph.fit(df, duration_col='DFS', event_col='event_DFS')

# Perform the proportional hazards assumption test
results = proportional_hazard_test(cph, df)
print(results)

<lifelines.CoxPHFitter: fitted with 139 total observations, 71 right-censored observations>

<lifelines.StatisticalResult: proportional_hazard_test>
    time_transform = rank
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 139 total observations, 71 right-censored observations>
         test_name = proportional_hazard_test

---
                                                       test_statistic      p  -log2(p)
LBP_003_CT                                                       0.06   0.81      0.31
LBP_003_PET                                                      0.76   0.38      1.38
LBP_012_CT                                                       0.13   0.72      0.48
LBP_012_PET                                                      0.27   0.60      0.73
LBP_021_CT                                                       0.56   0.45      1.14
LBP_021_PET                                                      0.00   0.97      0.04
LBP_030_CT                                                       0.04   0.83      0.26
LB

In [34]:
# Access the summary table
summary_table = results.summary

# Filter columns based on p-values lower than 0.05
significant_columns = summary_table[summary_table['p'] < 0.05].index

# Display significant columns
print("Columns with p-values < 0.05:")
print(significant_columns)

Columns with p-values < 0.05:
Index(['charlson', 'glszm_LargeAreaEmphasis_CT_c16',
       'glszm_LargeAreaLowGrayLevelEmphasis_CT_c16',
       'glszm_ZoneVariance_CT_c16', 'ngtdm_Busyness_d_1_PET_b2'],
      dtype='object')


## Test dataset: MAASTRO 

In [35]:
(MAASTRO_D3['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [36]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [37]:
# Need to choose patient_id from MAASTRO_D3 in response_MAASTRO
data = list(MAASTRO_D3['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 
response_MAASTRO

,patient_id,OS,OS_event,LRC,LRC_event,DFS,DFS_event
0,1,62.43,0.0,62.43,0.0,62.43,0.0
1,2,60.00,0.0,60.00,0.0,60.00,0.0
2,3,44.43,1.0,8.83,1.0,8.83,1.0
3,4,37.20,1.0,19.37,0.0,19.73,1.0
5,6,59.23,0.0,59.23,0.0,59.23,0.0
...,...,...,...,...,...,...,...
109,110,19.00,1.0,13.27,1.0,13.27,1.0
110,111,85.87,1.0,82.83,0.0,85.87,1.0
111,112,42.87,0.0,42.87,0.0,42.87,0.0
112,113,58.93,0.0,58.93,0.0,58.93,0.0


In [38]:
# Merge MAASTRO_D3 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D3, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'OS_event', 'LRC', 'LRC_event'])]
clinical_test

,patient_id,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,DFS,DFS_event
0,1,55,0,0,1,0,0,1,1,1,...,0.028808,0.837387,0.000026,0.001705,0.122209,0.000026,0.008291,0.000341,62.43,0.0
1,2,55,0,0,1,0,0,0,0,0,...,0.049615,0.806842,0.000167,0.002958,0.128976,0.000167,0.008148,0.000335,60.00,0.0
2,3,55,0,0,1,0,0,0,0,1,...,0.019514,0.830272,0.000057,0.001831,0.137282,0.000000,0.008641,0.000229,8.83,1.0
3,4,61,1,0,0,0,1,1,0,1,...,0.047155,0.760949,0.000000,0.003597,0.171595,0.000080,0.013187,0.000240,19.73,1.0
4,6,70,0,0,1,0,0,1,1,1,...,0.033990,0.824865,0.000000,0.002116,0.128134,0.000035,0.009379,0.000529,59.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,66,1,0,0,0,1,0,0,0,...,0.017489,0.834590,0.000000,0.001321,0.137194,0.000078,0.008706,0.000078,13.27,1.0
95,111,63,0,0,0,0,1,0,0,1,...,0.029979,0.821431,0.000000,0.000543,0.137620,0.000054,0.008907,0.000489,85.87,1.0
96,112,63,0,0,1,0,0,1,1,1,...,0.017354,0.859957,0.000000,0.000988,0.115099,0.000028,0.005700,0.000141,42.87,0.0
97,113,54,0,0,1,0,0,1,1,0,...,0.025399,0.847908,0.000000,0.001487,0.117654,0.000000,0.005759,0.000038,58.93,0.0


In [39]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [40]:
# Some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,DFS,DFS_event


In [41]:
# Set X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS'])]

# Set y 
y = clinical_train.loc[:, ['DFS', 'event_DFS']]

In [42]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [43]:
# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

X_train:  (139, 388)
y_train:  (139,)


In [44]:
# Change the name of a column 'DFS_event' in the clincial_test 
clinical_test.rename(columns = {'DFS_event' : 'event_DFS'}, inplace = True)

In [45]:
# Set X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'event_DFS'])]

# Set y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['DFS', 'event_DFS']]

# Change y_MAASTRO into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_DFS'], y_MAASTRO['DFS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

(99, 390)

# Yeo-Johnson Transformation

In [46]:
# Copy the original X for later 
original_X = X.copy()

In [48]:
# Standardize the data 
## Save the column and index 
X_columns = X.columns 
X_index = X.index

## Standardize the data but not the categorical columns
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']
### Separate the categorical and non-categorical columns
X_categorical = X[categorical_columns]
X_numeric = X.drop(categorical_columns, axis=1)
X_numeric_columns = X_numeric.columns
X_numeric_index = X_numeric.index

### Standardize non-categorical and then concat with the categorical
pt = PowerTransformer(method='yeo-johnson')
X_numeric_std = pt.fit_transform(X_numeric)
X_numeric_std = pd.DataFrame(X_numeric_std, columns=X_numeric_columns, index=X_numeric_index)
X_std = pd.concat([X_categorical, X_numeric_std], axis=1)

## Sort the order of the columns as it was in the clinical train 
X_std = X_std[original_X.columns]

In [49]:
# Standardize X_MAASTRO 
MAASTRO_new = X_MAASTRO.copy()
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns

MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = pt.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
MAASTRO_new = MAASTRO_new[X_new.columns]
MAASTRO_new_std = MAASTRO_new_std[MAASTRO_new.columns]

In [51]:
# Saving the data 
X_new = X 
X_new_std = X_std 
MAASTRO_new = X_MAASTRO 
MAASTRO_new_std = MAASTRO_new_std 

In [52]:
X_new

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,...,LBP_003_PET,LBP_012_PET,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET
0,54.238356,1,0,1,0,0,1,0.0,0,0.000000,...,0.000123,0.001172,0.029974,0.815962,0.000000,0.002035,0.140311,0.000062,0.009991,0.000370
1,54.539726,0,0,0,0,1,0,0.0,1,27.404795,...,0.000349,0.008383,0.059728,0.707300,0.000000,0.008732,0.191058,0.000349,0.023402,0.000699
2,59.019178,0,1,0,0,0,1,0.0,1,41.019178,...,0.000000,0.001759,0.039463,0.819828,0.000034,0.002518,0.126531,0.000000,0.009452,0.000414
3,70.726027,0,0,0,0,1,0,0.0,1,37.500000,...,0.000300,0.006892,0.061133,0.716212,0.000000,0.003296,0.192388,0.000000,0.018879,0.000899
4,67.865753,0,0,0,0,1,0,0.0,1,53.000000,...,0.000000,0.001993,0.058589,0.696493,0.000199,0.008968,0.202073,0.000399,0.029892,0.001395
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134,60.435616,0,0,1,0,0,1,1.0,0,0.000000,...,0.000000,0.001562,0.029444,0.804831,0.000000,0.001442,0.152626,0.000000,0.009734,0.000361
135,68.794521,0,0,1,0,0,1,1.0,0,0.000000,...,0.000039,0.001968,0.046451,0.792151,0.000000,0.004094,0.142778,0.000079,0.011810,0.000630
136,57.498630,0,0,1,0,0,1,1.0,1,39.498630,...,0.000000,0.000326,0.016316,0.832202,0.000000,0.000914,0.140582,0.000000,0.009137,0.000522
137,65.684932,0,0,1,0,0,1,1.0,1,71.527397,...,0.000000,0.001030,0.038862,0.787588,0.000000,0.003144,0.156640,0.000054,0.011978,0.000705


In [53]:
X_new_std

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,...,LBP_003_PET,LBP_012_PET,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET
0,-0.785036,1,0,1,0,0,1,0.0,0,-1.533345,...,1.042615,-0.340702,-0.305657,0.622355,-0.784694,-0.404309,-0.370546,-0.074611,-0.654022,-0.325914
1,-0.746998,0,0,0,0,1,0,0.0,1,0.397233,...,1.929439,1.907148,0.939011,-1.489676,-0.784694,1.826912,1.188847,1.689445,1.560347,0.852496
2,-0.175256,0,1,0,0,0,1,0.0,1,0.830437,...,-0.799521,0.082511,0.147926,0.741227,0.236612,-0.110511,-1.026597,-1.065926,-0.824361,-0.125568
3,1.371631,0,0,0,0,1,0,0.0,1,0.727938,...,1.846691,1.751709,0.986180,-1.383365,-0.784694,0.301039,1.216712,-1.065926,1.122247,1.296177
4,0.987008,0,0,0,0,1,0,0.0,1,1.144253,...,-0.799521,0.230152,0.900131,-1.607406,1.784568,1.860059,1.405024,1.782787,1.919501,1.900963
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134,0.007949,0,0,1,0,0,1,1.0,0,-1.533345,...,-0.799521,-0.050750,-0.332827,0.301629,-0.784694,-0.811377,0.115613,-1.065926,-0.733951,-0.371465
135,1.111443,0,0,1,0,0,1,1.0,0,-1.533345,...,-0.000653,0.215135,0.445866,-0.027763,-0.784694,0.654869,-0.266178,0.138716,-0.145951,0.659834
136,-0.370651,0,0,1,0,0,1,1.0,1,0.786825,...,-0.799521,-1.105445,-1.073694,1.149438,-0.784694,-1.222412,-0.358889,-1.065926,-0.928256,0.305804
137,0.696583,0,0,1,0,0,1,1.0,1,1.554125,...,-0.799521,-0.455330,0.120914,-0.137599,-0.784694,0.225798,0.256551,-0.175482,-0.103535,0.868321


In [54]:
MAASTRO_new

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,...,LBP_003_PET,LBP_012_PET,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET
0,55,0,0,1,0,0,1,1,1,0,...,0.000026,0.001181,0.028808,0.837387,0.000026,0.001705,0.122209,0.000026,0.008291,0.000341
1,55,0,0,1,0,0,0,0,0,20,...,0.000056,0.002735,0.049615,0.806842,0.000167,0.002958,0.128976,0.000167,0.008148,0.000335
2,55,0,0,1,0,0,0,0,1,6,...,0.000286,0.001888,0.019514,0.830272,0.000057,0.001831,0.137282,0.000000,0.008641,0.000229
3,61,1,0,0,0,1,1,0,1,45,...,0.000160,0.003037,0.047155,0.760949,0.000000,0.003597,0.171595,0.000080,0.013187,0.000240
4,70,0,0,1,0,0,1,1,1,59,...,0.000071,0.000881,0.033990,0.824865,0.000000,0.002116,0.128134,0.000035,0.009379,0.000529
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,66,1,0,0,0,1,0,0,0,55,...,0.000000,0.000544,0.017489,0.834590,0.000000,0.001321,0.137194,0.000078,0.008706,0.000078
95,63,0,0,0,0,1,0,0,1,174,...,0.000109,0.000869,0.029979,0.821431,0.000000,0.000543,0.137620,0.000054,0.008907,0.000489
96,63,0,0,1,0,0,1,1,1,0,...,0.000000,0.000734,0.017354,0.859957,0.000000,0.000988,0.115099,0.000028,0.005700,0.000141
97,54,0,0,1,0,0,1,1,0,0,...,0.000114,0.001640,0.025399,0.847908,0.000000,0.001487,0.117654,0.000000,0.005759,0.000038


In [55]:
MAASTRO_new_std

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,...,LBP_003_PET,LBP_012_PET,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET
0,-0.688798,0,0,1,0,0,1,1,1,-1.533345,...,-0.238970,-0.333758,-0.365666,1.333829,0.033184,-0.624049,-1.260759,-0.597387,-1.225380,-0.467062
1,-0.688798,0,0,1,0,0,0,0,0,0.104884,...,0.263801,0.633163,0.571622,0.357306,1.682719,0.130792,-0.900442,0.946684,-1.278036,-0.498297
2,-0.688798,0,0,1,0,0,0,0,1,-0.713198,...,1.817599,0.165484,-0.880459,1.082861,0.693043,-0.538327,-0.503809,-1.065926,-1.099242,-1.088084
3,0.081262,1,0,0,0,1,1,0,1,0.940197,...,1.314752,0.772266,0.474304,-0.697526,-0.784694,0.441636,0.716866,0.152800,0.179520,-1.022290
4,1.273609,0,0,1,0,0,1,1,1,1.285347,...,0.471935,-0.580655,-0.106360,0.902157,-0.784694,-0.353231,-0.943409,-0.453142,-0.848088,0.330123
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,0.738387,1,0,0,0,1,0,0,0,1.192311,...,-0.799521,-0.888554,-1.001790,1.233343,-0.784694,-0.901046,-0.507763,0.126817,-1.076381,-2.149148
95,0.342484,0,0,0,0,1,0,0,1,3.094727,...,0.907497,-0.591523,-0.305416,0.791663,-0.784694,-1.541116,-0.488635,-0.173978,-1.006544,0.181649
96,0.342484,0,0,1,0,0,1,1,1,-1.533345,...,-0.799521,-0.711549,-1.010039,2.238796,-0.784694,-1.161970,-1.679560,-0.565019,-2.317743,-1.669244
97,-0.815082,0,0,1,0,0,1,1,0,-1.533345,...,0.962590,0.002735,-0.546695,1.734121,-0.784694,-0.778405,-1.524056,-1.065926,-2.289503,-2.477923


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [56]:
# Setting the y format for skf below  
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-16 20:49:40,550] A new study created in memory with name: no-name-465d94ad-28cd-45d5-8e15-4695103d03fc
python(33114) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


  0%|          | 0/1 [00:00<?, ?it/s]

[W 2024-04-16 20:49:43,982] Trial 0 failed with parameters: {} because of the following error: LinAlgError('Matrix is singular.').
Traceback (most recent call last):
  File "/Users/minjeongcheon/opt/anaconda3/lib/python3.9/site-packages/optuna/study/_optimize.py", line 200, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/64/jkqp6xyx2hj50dmd2pqfm3780000gn/T/ipykernel_31603/3148227347.py", line 62, in objective
    model.fit(X_train_std, y_train)
  File "/Users/minjeongcheon/opt/anaconda3/lib/python3.9/site-packages/sksurv/linear_model/coxph.py", line 449, in fit
    delta = solve(
  File "/Users/minjeongcheon/opt/anaconda3/lib/python3.9/site-packages/scipy/linalg/_basic.py", line 220, in solve
    _solve_check(n, info)
  File "/Users/minjeongcheon/opt/anaconda3/lib/python3.9/site-packages/scipy/linalg/_basic.py", line 29, in _solve_check
    raise LinAlgError('Matrix is singular.')
numpy.linalg.LinAlgError: Matrix is singular.
[W 2024-04-16 20:49:43,992] Trial 0 fai

LinAlgError: Matrix is singular.

In [59]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

ValueError: No trials are completed yet.

In [60]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

ValueError: No trials are completed yet.

#### Test

In [61]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [62]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

LinAlgError: Matrix is singular.

In [63]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [64]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

NameError: name 'c_index' is not defined

### 2. CoxnetSurvivalAnalysis - Ridge

#### Train

In [65]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 20:49:57,834] A new study created in memory with name: no-name-a682c9c4-bd26-4db7-b9c7-411e96522584


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.5756972111553785
Fold 2 C-index: 0.7073643410852714
Fold 3 C-index: 0.5702127659574469
Fold 4 C-index: 0.6653992395437263


[I 2024-04-16 20:50:08,080] A new study created in memory with name: no-name-f76268ca-2509-4569-a91f-237e1900b275


Fold 5 C-index: 0.6566523605150214
[I 2024-04-16 20:50:08,063] Trial 0 finished with value: 0.635065183651369 and parameters: {}. Best is trial 0 with value: 0.635065183651369.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.635065183651369], datetime_start=datetime.datetime(2024, 4, 16, 20, 49, 57, 871851), datetime_complete=datetime.datetime(2024, 4, 16, 20, 50, 8, 62951), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.635065183651369


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.24724709663544278
Fold 2 IBS: 0.23203988184644345
Fold 3 IBS: 0.22898186778522736
Fold 4 IBS: 0.24197476559442144
Fold 5 IBS: 0.22939558964342896
[I 2024-04-16 20:50:20,794] Trial 0 finished with value: 0.23592784030099279 and parameters: {}. Best is trial 0 with value: 0.23592784030099279.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.23592784030099279], datetime_start=datetime.datetime(2024, 4, 16, 20, 50, 8, 158709), datetime_complete=datetime.datetime(2024, 4, 16, 20, 50, 20, 793610), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.23592784030099279


In [66]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [67]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.635
train_ibs:  0.236


#### Test

In [68]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [69]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.53


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.229


In [70]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [71]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 20:50:21,121] A new study created in memory with name: no-name-73a33ba5-e856-4c27-9d7e-c3a5255bd1d5


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.5737051792828686
Fold 2 C-index: 0.6395348837209303
Fold 3 C-index: 0.4723404255319149
Fold 4 C-index: 0.623574144486692


[I 2024-04-16 20:50:35,487] A new study created in memory with name: no-name-c63a5ae8-e1bb-4960-8eb8-e945c00d0df3


Fold 5 C-index: 0.6051502145922747
[I 2024-04-16 20:50:35,471] Trial 0 finished with value: 0.5828609695229361 and parameters: {}. Best is trial 0 with value: 0.5828609695229361.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.5828609695229361], datetime_start=datetime.datetime(2024, 4, 16, 20, 50, 21, 151201), datetime_complete=datetime.datetime(2024, 4, 16, 20, 50, 35, 471518), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.5828609695229361


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.39850056002671524
Fold 2 IBS: 0.28579825945985843
Fold 3 IBS: 0.40623214528565893
Fold 4 IBS: 0.35516159839951
Fold 5 IBS: 0.3204294572240529
[I 2024-04-16 20:50:50,424] Trial 0 finished with value: 0.3532244040791591 and parameters: {}. Best is trial 0 with value: 0.3532244040791591.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.3532244040791591], datetime_start=datetime.datetime(2024, 4, 16, 20, 50, 35, 521987), datetime_complete=datetime.datetime(2024, 4, 16, 20, 50, 50, 423562), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.3532244040791591


In [72]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [73]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.583
train_ibs:  0.353


#### Test

In [74]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [75]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.491


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.457


In [76]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [77]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 20:50:52,514] A new study created in memory with name: no-name-ab1b88e5-65e7-4213-bd4b-24589fd74fe1


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5617529880478087
Fold 2 C-index: 0.6511627906976745
Fold 3 C-index: 0.5106382978723404
Fold 4 C-index: 0.6083650190114068
Fold 5 C-index: 0.5836909871244635
[I 2024-04-16 20:51:03,626] Trial 0 finished with value: 0.5831220165507387 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.5831220165507387.
Fold 1 C-index: 0.5219123505976095
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5234042553191489
Fold 4 C-index: 0.5703422053231939
Fold 5 C-index: 0.5536480686695279
[I 2024-04-16 20:51:20,443] Trial 1 finished with value: 0.5733962597028263 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.5831220165507387.
Fold 1 C-index: 0.5099601593625498
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5404255319148936
Fold 4 C-index: 0.5817490494296578
Fold 5 C-index: 0.5579399141630901
[I 2024-04-16 20:51:32,528] Trial 2 finished with value: 0.5775498146949685 and parameters: {'l1_ratio': 0.22692876841884668}.

Fold 1 C-index: 0.5258964143426295
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5234042553191489
Fold 4 C-index: 0.5703422053231939
Fold 5 C-index: 0.5536480686695279
[I 2024-04-16 20:56:56,186] Trial 24 finished with value: 0.5741930724518303 and parameters: {'l1_ratio': 0.2985516395803664}. Best is trial 23 with value: 0.6380665972349431.
Fold 1 C-index: 0.49800796812749004
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.5741444866920152
Fold 5 C-index: 0.6738197424892703
[I 2024-04-16 20:57:05,039] Trial 25 finished with value: 0.6010697487146002 and parameters: {'l1_ratio': 0.1613975971413524}. Best is trial 23 with value: 0.6380665972349431.
Fold 1 C-index: 0.5737051792828686
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6920152091254753
Fold 5 C-index: 0.6652360515021459
[I 2024-04-16 20:57:15,895] Trial 26 finished with value: 0.6380665972349431 and parameters: {'l1_ratio': 0.07156750350022

Fold 5 C-index: 0.5665236051502146
[I 2024-04-16 21:02:39,651] Trial 47 finished with value: 0.5710363936875751 and parameters: {'l1_ratio': 0.46157256171761685}. Best is trial 34 with value: 0.6389249663336555.
Fold 1 C-index: 0.5737051792828686
Fold 2 C-index: 0.6395348837209303
Fold 3 C-index: 0.46808510638297873
Fold 4 C-index: 0.6121673003802282
Fold 5 C-index: 0.6008583690987125
[I 2024-04-16 21:02:48,570] Trial 48 finished with value: 0.5788701677731437 and parameters: {'l1_ratio': 0.908082736959313}. Best is trial 34 with value: 0.6389249663336555.
Fold 1 C-index: 0.545816733067729
Fold 2 C-index: 0.6511627906976745
Fold 3 C-index: 0.5063829787234042
Fold 4 C-index: 0.6083650190114068
Fold 5 C-index: 0.5665236051502146
[I 2024-04-16 21:02:59,871] Trial 49 finished with value: 0.5756502253300858 and parameters: {'l1_ratio': 0.5349713871649028}. Best is trial 34 with value: 0.6389249663336555.
Fold 1 C-index: 0.5776892430278885
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0

...

  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 IBS: 0.3966465644914146
Fold 2 IBS: 0.28637185420169997
Fold 3 IBS: 0.4025192458137644
Fold 4 IBS: 0.32580492906293385
Fold 5 IBS: 0.29932384811304946
[I 2024-04-16 21:10:19,531] Trial 0 finished with value: 0.3421332883365724 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.3421332883365724.
Fold 1 IBS: 0.40259804979747177
Fold 2 IBS: 0.2265618087310044
Fold 3 IBS: 0.36724961425493335
Fold 4 IBS: 0.33390963573464383
Fold 5 IBS: 0.2705009693565652
[I 2024-04-16 21:10:28,911] Trial 1 finished with value: 0.3201640155749237 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.3201640155749237.
Fold 1 IBS: 0.4026433726024702
Fold 2 IBS: 0.22741753924120442
Fold 3 IBS: 0.35508264627336156
Fold 4 IBS: 0.3305784833519445
Fold 5 IBS: 0.2672392448991321
[I 2024-04-16 21:10:38,756] Trial 2 finished with value: 0.31659225727362256 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 2 with value: 0.31659225727362256.
F

Fold 1 IBS: 0.38371001165015334
Fold 2 IBS: 0.22991305573774998
Fold 3 IBS: 0.22878890533246202
Fold 4 IBS: 0.23805110419288075
Fold 5 IBS: 0.22694636185590233
[I 2024-04-16 21:14:25,759] Trial 25 finished with value: 0.2614818877538297 and parameters: {'l1_ratio': 0.08890608406899088}. Best is trial 12 with value: 0.23515110302320735.
Fold 1 IBS: 0.40286382034066365
Fold 2 IBS: 0.22729127829046636
Fold 3 IBS: 0.35709880984522213
Fold 4 IBS: 0.33131391100887675
Fold 5 IBS: 0.26791531956834874
[I 2024-04-16 21:14:35,188] Trial 26 finished with value: 0.31729662781071555 and parameters: {'l1_ratio': 0.23521758130887002}. Best is trial 12 with value: 0.23515110302320735.
Fold 1 IBS: 0.3983048574647202
Fold 2 IBS: 0.29384151511158013
Fold 3 IBS: 0.41048543674537785
Fold 4 IBS: 0.33012863520732916
Fold 5 IBS: 0.31169105886780124
[I 2024-04-16 21:14:46,763] Trial 27 finished with value: 0.3488903006793617 and parameters: {'l1_ratio': 0.8345729974660353}. Best is trial 12 with value: 0.235151

Fold 1 IBS: 0.39768104529022175
Fold 2 IBS: 0.27478843392280616
Fold 3 IBS: 0.39610715327059925
Fold 4 IBS: 0.32930852997155785
Fold 5 IBS: 0.28466923297814606
[I 2024-04-16 21:29:17,955] Trial 50 finished with value: 0.3365108790866662 and parameters: {'l1_ratio': 0.556147470734502}. Best is trial 30 with value: 0.2338861317669947.
Fold 1 IBS: 0.24527167584214757
Fold 2 IBS: 0.23043201333842706
Fold 3 IBS: 0.22882959418766094
Fold 4 IBS: 0.23886096933818587
Fold 5 IBS: 0.2275108188111229
[I 2024-04-16 21:29:24,367] Trial 51 finished with value: 0.23418101430350885 and parameters: {'l1_ratio': 0.06521384802113693}. Best is trial 30 with value: 0.2338861317669947.
Fold 1 IBS: 0.24563902210807548
Fold 2 IBS: 0.23076318528471024
Fold 3 IBS: 0.228857861655577
Fold 4 IBS: 0.23942629816028696
Fold 5 IBS: 0.22788081987002892
[I 2024-04-16 21:29:29,964] Trial 52 finished with value: 0.2345134374157357 and parameters: {'l1_ratio': 0.05081950410384198}. Best is trial 30 with value: 0.23388613176

Fold 1 IBS: 0.24502298791627394
Fold 2 IBS: 0.23019984637955831
Fold 3 IBS: 0.22881083058401447
Fold 4 IBS: 0.23848030996038092
Fold 5 IBS: 0.22725595943412547
[I 2024-04-16 21:32:37,088] Trial 75 finished with value: 0.23395398685487062 and parameters: {'l1_ratio': 0.07563581400941731}. Best is trial 65 with value: 0.2338114901918404.
Fold 1 IBS: 0.39470143336903885
Fold 2 IBS: 0.22905991597084543
Fold 3 IBS: 0.22873248532776408
Fold 4 IBS: 0.2369097692969695
Fold 5 IBS: 0.2260452207998657
[I 2024-04-16 21:32:43,736] Trial 76 finished with value: 0.2630897649528967 and parameters: {'l1_ratio': 0.13120285166575224}. Best is trial 65 with value: 0.2338114901918404.
Fold 1 IBS: 0.4022362600841385
Fold 2 IBS: 0.2263144117931058
Fold 3 IBS: 0.37046848132770543
Fold 4 IBS: 0.3338040954427133
Fold 5 IBS: 0.27147653458970394
[I 2024-04-16 21:32:51,672] Trial 77 finished with value: 0.32085995664747335 and parameters: {'l1_ratio': 0.3048122805041498}. Best is trial 65 with value: 0.23381149019

In [78]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [79]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.641
train_ibs:  0.234


#### Test

In [80]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [81]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.02546295954223466)

test_cindex : 0.527


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.08286735522305856)

test_ibs:  0.229


In [82]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [83]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-16 21:36:21,970] A new study created in memory with name: no-name-393da273-6c70-48cb-a6b7-e41ead23b7a3


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6294820717131474
Fold 2 C-index: 0.751937984496124
Fold 3 C-index: 0.6680851063829787
Fold 4 C-index: 0.7376425855513308
Fold 5 C-index: 0.6566523605150214
[I 2024-04-16 21:36:48,796] Trial 0 finished with value: 0.6887600217317204 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.6887600217317204.
Fold 1 C-index: 0.5418326693227091
Fold 2 C-index: 0.6550387596899225
Fold 3 C-index: 0.7191489361702128
Fold 4 C-index: 0.6349809885931559
Fold 5 C-index: 0.5321888412017167
[I 2024-04-16 21:36:53,431] Trial 1 finished with value: 0.6166380389955435 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, '

Fold 1 C-index: 0.4203187250996016
Fold 2 C-index: 0.6996124031007752
Fold 3 C-index: 0.7340425531914894
Fold 4 C-index: 0.564638783269962
Fold 5 C-index: 0.6330472103004292
[I 2024-04-16 21:38:16,868] Trial 16 finished with value: 0.6103319349924515 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 2, 'min_samples_leaf': 18, 'max_depth': 6, 'n_estimators': 4, 'oob_score': True, 'max_samples': 0.8500299963488761, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.3855568975046046, 'warm_start': True}. Best is trial 14 with value: 0.7054382711677689.
Fold 1 C-index: 0.49800796812749004
Fold 2 C-index: 0.810077519379845
Fold 3 C-index: 0.851063829787234
Fold 4 C-index: 0.7642585551330798
Fold 5 C-index: 0.7467811158798283
[I 2024-04-16 21:38:17,877] Trial 17 finished with value: 0.7340377976614955 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 8, 'min_samples_leaf': 16, 'max_depth': 7, 'n_estimators': 96, 'oob_score': True, 'max_samples': 0.9816256083986692, 'ma

Fold 1 C-index: 0.5139442231075697
Fold 2 C-index: 0.8023255813953488
Fold 3 C-index: 0.8723404255319149
Fold 4 C-index: 0.7642585551330798
Fold 5 C-index: 0.7939914163090128
[I 2024-04-16 21:38:41,736] Trial 31 finished with value: 0.7493720402953852 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 18, 'min_samples_leaf': 8, 'max_depth': 13, 'n_estimators': 239, 'oob_score': True, 'max_samples': 0.6565648650889025, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.14061763521617526, 'warm_start': True}. Best is trial 27 with value: 0.7847011373516671.
Fold 1 C-index: 0.5219123505976095
Fold 2 C-index: 0.8062015503875969
Fold 3 C-index: 0.8808510638297873
Fold 4 C-index: 0.8136882129277566
Fold 5 C-index: 0.8454935622317596
[I 2024-04-16 21:38:43,484] Trial 32 finished with value: 0.7736293479949021 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 17, 'min_samples_leaf': 7, 'max_depth': 12, 'n_estimators': 241, 'oob_score': True, 'max_samples': 0.717337635186249

...

Fold 1 IBS: 0.22149490022552124
Fold 2 IBS: 0.18466865634717505
Fold 3 IBS: 0.23548539432841317
Fold 4 IBS: 0.2113697424267566
Fold 5 IBS: 0.2119677581906565
[I 2024-04-16 22:08:14,837] Trial 32 finished with value: 0.2129972903037045 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 11, 'min_samples_leaf': 8, 'max_depth': 14, 'n_estimators': 294, 'oob_score': True, 'max_samples': 0.9673759674409311, 'max_features': None, 'min_weight_fraction_leaf': 0.27417727195073677}. Best is trial 32 with value: 0.2129972903037045.
Fold 1 IBS: 0.2567229873354098
Fold 2 IBS: 0.18854079292396503
Fold 3 IBS: 0.2249457788925835
Fold 4 IBS: 0.22508129343804778
Fold 5 IBS: 0.21386415001842546
[I 2024-04-16 22:08:47,016] Trial 33 finished with value: 0.2218310005216863 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 13, 'min_samples_leaf': 13, 'max_depth': 13, 'n_estimators': 304, 'oob_score': True, 'max_samples': 0.9964442664551478, 'max_features': None, 'min_weight_fraction_leaf': 0

Fold 1 IBS: 0.23233061602902896
Fold 2 IBS: 0.20470776398952695
Fold 3 IBS: 0.24473722549758978
Fold 4 IBS: 0.23554082329183057
Fold 5 IBS: 0.22266798632628182
[I 2024-04-16 22:11:11,177] Trial 48 finished with value: 0.22799688302685164 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 9, 'min_samples_leaf': 9, 'max_depth': 14, 'n_estimators': 277, 'oob_score': True, 'max_samples': 0.7486071388218823, 'max_features': None, 'min_weight_fraction_leaf': 0.3316489763494004}. Best is trial 43 with value: 0.2123917039278164.
Fold 1 IBS: 0.22140751210407167
Fold 2 IBS: 0.18446093935852728
Fold 3 IBS: 0.23819663142832964
Fold 4 IBS: 0.21304733351095642
Fold 5 IBS: 0.21354093321412873
[I 2024-04-16 22:11:33,770] Trial 49 finished with value: 0.21413066992320276 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 18, 'min_samples_leaf': 20, 'max_depth': 11, 'n_estimators': 373, 'oob_score': True, 'max_samples': 0.9104681524496127, 'max_features': None, 'min_weight_fraction_leaf

Fold 1 IBS: 0.2365430487443625
Fold 2 IBS: 0.18238280317098843
Fold 3 IBS: 0.2322276544017584
Fold 4 IBS: 0.21380685041046066
Fold 5 IBS: 0.21115528132467798
[I 2024-04-16 22:16:41,628] Trial 64 finished with value: 0.21522312761044962 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 18, 'min_samples_leaf': 17, 'max_depth': 9, 'n_estimators': 405, 'oob_score': True, 'max_samples': 0.9369908825823273, 'max_features': None, 'min_weight_fraction_leaf': 0.19119487575645203}. Best is trial 43 with value: 0.2123917039278164.
Fold 1 IBS: 0.2545794604503248
Fold 2 IBS: 0.18515187912067008
Fold 3 IBS: 0.22861527937785475
Fold 4 IBS: 0.22050482646274178
Fold 5 IBS: 0.2129134229864224
[I 2024-04-16 22:17:14,329] Trial 65 finished with value: 0.22035297367960277 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 16, 'min_samples_leaf': 15, 'max_depth': 6, 'n_estimators': 362, 'oob_score': True, 'max_samples': 0.9998359499913109, 'max_features': None, 'min_weight_fraction_leaf': 0

Fold 1 IBS: 0.2511057207255618
Fold 2 IBS: 0.21572593535559278
Fold 3 IBS: 0.2149073004231141
Fold 4 IBS: 0.2472468833843753
Fold 5 IBS: 0.24247425822381166
[I 2024-04-16 22:21:29,895] Trial 80 finished with value: 0.23429201962249113 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 20, 'min_samples_leaf': 16, 'max_depth': 12, 'n_estimators': 335, 'oob_score': True, 'max_samples': 0.3618038809495957, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.15390327148267055}. Best is trial 43 with value: 0.2123917039278164.
Fold 1 IBS: 0.22731065610063553
Fold 2 IBS: 0.18233470119562936
Fold 3 IBS: 0.23388585350331287
Fold 4 IBS: 0.21305351886000728
Fold 5 IBS: 0.21153134197254575
[I 2024-04-16 22:21:50,180] Trial 81 finished with value: 0.21362321432642614 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 17, 'min_samples_leaf': 18, 'max_depth': 11, 'n_estimators': 318, 'oob_score': True, 'max_samples': 0.9436804929256887, 'max_features': None, 'min_weight_fraction_l

Fold 1 IBS: 0.21975345965751325
Fold 2 IBS: 0.1826687616539912
Fold 3 IBS: 0.23593232323623828
Fold 4 IBS: 0.21692323202986982
Fold 5 IBS: 0.2157946933853038
[I 2024-04-16 22:25:46,679] Trial 96 finished with value: 0.21421449399258327 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 9, 'min_samples_leaf': 20, 'max_depth': 12, 'n_estimators': 296, 'oob_score': True, 'max_samples': 0.8401991280225705, 'max_features': None, 'min_weight_fraction_leaf': 0.20291905202477475}. Best is trial 85 with value: 0.21229918040313978.
Fold 1 IBS: 0.22595559817931396
Fold 2 IBS: 0.18811290557691393
Fold 3 IBS: 0.23790732230802805
Fold 4 IBS: 0.22652570350197412
Fold 5 IBS: 0.21929538367243218
[I 2024-04-16 22:26:02,326] Trial 97 finished with value: 0.21955938264773245 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 7, 'min_samples_leaf': 19, 'max_depth': 15, 'n_estimators': 270, 'oob_score': True, 'max_samples': 0.9001142343235446, 'max_features': None, 'min_weight_fraction_leaf'

In [84]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [85]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.847
train_ibs:  0.212


#### Test

In [86]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

In [87]:
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])
 
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=16, max_features='auto', max_leaf_nodes=14,
                     max_samples=0.9873533850424658, min_samples_leaf=4,
                     min_samples_split=8,
                     min_weight_fraction_leaf=0.04287202412707603,
                     n_estimators=441, random_state=123, warm_start=True)

test_cindex:  0.54


RandomSurvivalForest(max_depth=13, max_features=None, max_leaf_nodes=11,
                     max_samples=0.9290276329324785, min_samples_leaf=20,
                     min_samples_split=9,
                     min_weight_fraction_leaf=0.20767416366171196,
                     n_estimators=268, oob_score=True, random_state=123)

test_ibs:  0.262


In [88]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [89]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [90]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 22:26:38,298] A new study created in memory with name: no-name-2f9503b6-9730-421f-982f-6c84fe9fba58


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.50199203187251
Fold 2 C-index: 0.7635658914728682
Fold 3 C-index: 0.8425531914893617
Fold 4 C-index: 0.7376425855513308
Fold 5 C-index: 0.7381974248927039
[I 2024-04-16 22:26:40,073] Trial 0 finished with value: 0.716790225055755 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.716790225055755.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 22:26:42,397] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531}. B

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 22:27:06,135] Trial 16 finished with value: 0.5 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 14, 'min_samples_leaf': 20, 'max_depth': 4, 'n_estimators': 231, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.6778661833429002, 'min_weight_fraction_leaf': 0.35612645124522524}. Best is trial 15 with value: 0.7572849484519277.
Fold 1 C-index: 0.5697211155378487
Fold 2 C-index: 0.7209302325581395
Fold 3 C-index: 0.8170212765957446
Fold 4 C-index: 0.7224334600760456
Fold 5 C-index: 0.6995708154506438
[I 2024-04-16 22:27:06,618] Trial 17 finished with value: 0.7059353800436845 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 11, 'min_samples_leaf': 14, 'max_depth': 9, 'n_estimators': 100, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.3861349103797888, 'min_weight_fraction_leaf': 0.0896057534547

Fold 1 C-index: 0.6055776892430279
Fold 2 C-index: 0.7635658914728682
Fold 3 C-index: 0.8723404255319149
Fold 4 C-index: 0.7414448669201521
Fold 5 C-index: 0.7424892703862661
[I 2024-04-16 22:27:16,310] Trial 31 finished with value: 0.7450836287108459 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 11, 'min_samples_leaf': 8, 'max_depth': 1, 'n_estimators': 89, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.6115041201105692, 'min_weight_fraction_leaf': 0.07529457384788989}. Best is trial 24 with value: 0.7999244832738667.
Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.8217054263565892
Fold 3 C-index: 0.8765957446808511
Fold 4 C-index: 0.8098859315589354
Fold 5 C-index: 0.8240343347639485
[I 2024-04-16 22:27:16,715] Trial 32 finished with value: 0.7891534508186784 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 12, 'min_samples_leaf': 6, 'max_depth': 2, 'n_estimators': 106, 'oob_score': False, 'warm_start': True, 'max_features'

...

Fold 1 IBS: 0.24189951983836425
Fold 2 IBS: 0.22116291305449884
Fold 3 IBS: 0.21713072638394984
Fold 4 IBS: 0.24023416048014104
Fold 5 IBS: 0.22691624187044754
[I 2024-04-16 22:43:05,381] Trial 30 finished with value: 0.2294687123254803 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 8, 'min_samples_leaf': 7, 'max_depth': 5, 'n_estimators': 180, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.5988223393644435, 'min_weight_fraction_leaf': 0.20241101766143602}. Best is trial 15 with value: 0.21555098369824366.
Fold 1 IBS: 0.24364620279536037
Fold 2 IBS: 0.2205505651382432
Fold 3 IBS: 0.21184589055027794
Fold 4 IBS: 0.24142198209294274
Fold 5 IBS: 0.2268782233658812
[I 2024-04-16 22:43:06,351] Trial 31 finished with value: 0.22886857278854106 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 13, 'min_samples_leaf': 11, 'max_depth': 8, 'n_estimators': 162, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0

Fold 1 IBS: 0.24970180838617465
Fold 2 IBS: 0.20711640796461644
Fold 3 IBS: 0.210622445605441
Fold 4 IBS: 0.2398038048480638
Fold 5 IBS: 0.23687430367329662
[I 2024-04-16 22:44:16,093] Trial 45 finished with value: 0.22882375409551847 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 19, 'min_samples_leaf': 6, 'max_depth': 13, 'n_estimators': 481, 'oob_score': True, 'warm_start': False, 'max_features': 'sqrt', 'max_samples': 0.6241818506764952, 'min_weight_fraction_leaf': 0.003649071529300437}. Best is trial 41 with value: 0.2125643090401542.
Fold 1 IBS: 0.23201805390304453
Fold 2 IBS: 0.20002733588165444
Fold 3 IBS: 0.19908476397700547
Fold 4 IBS: 0.22074033459452136
Fold 5 IBS: 0.21494542822340232
[I 2024-04-16 22:44:22,474] Trial 46 finished with value: 0.2133631833159256 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 17, 'min_samples_leaf': 2, 'max_depth': 15, 'n_estimators': 419, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.55

Fold 1 IBS: 0.24516656524539981
Fold 2 IBS: 0.21062905264741455
Fold 3 IBS: 0.20928713648563754
Fold 4 IBS: 0.24177268946133224
Fold 5 IBS: 0.23349939521162735
[I 2024-04-16 22:45:28,133] Trial 60 finished with value: 0.2280709678102823 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 13, 'min_samples_leaf': 10, 'max_depth': 12, 'n_estimators': 382, 'oob_score': True, 'warm_start': False, 'max_features': 'sqrt', 'max_samples': 0.7740796078428375, 'min_weight_fraction_leaf': 0.004254061138558238}. Best is trial 59 with value: 0.21106902851963083.
Fold 1 IBS: 0.23239287585025117
Fold 2 IBS: 0.19481155216054105
Fold 3 IBS: 0.19221809734433284
Fold 4 IBS: 0.21072513374212
Fold 5 IBS: 0.21430420045958357
[I 2024-04-16 22:45:47,850] Trial 61 finished with value: 0.20889037191136572 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 14, 'min_samples_leaf': 9, 'max_depth': 12, 'n_estimators': 258, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.

Fold 1 IBS: 0.23374241288259937
Fold 2 IBS: 0.19930439672774886
Fold 3 IBS: 0.18717738406829976
Fold 4 IBS: 0.20823176235963525
Fold 5 IBS: 0.21501371316102338
[I 2024-04-16 23:04:10,213] Trial 75 finished with value: 0.20869393383986132 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 11, 'min_samples_leaf': 10, 'max_depth': 12, 'n_estimators': 435, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.9528867121166885, 'min_weight_fraction_leaf': 0.0012590377645196535}. Best is trial 73 with value: 0.20839397477875155.
Fold 1 IBS: 0.24525891729558977
Fold 2 IBS: 0.21896003468791347
Fold 3 IBS: 0.21598508493564658
Fold 4 IBS: 0.24394612844374725
Fold 5 IBS: 0.2322350643210711
[I 2024-04-16 23:04:14,065] Trial 76 finished with value: 0.23127704593679366 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 11, 'min_samples_leaf': 13, 'max_depth': 11, 'n_estimators': 465, 'oob_score': True, 'warm_start': False, 'max_features': 'log2', 'max_samples

Fold 1 IBS: 0.24308229605873613
Fold 2 IBS: 0.20243085243083073
Fold 3 IBS: 0.20257300733372727
Fold 4 IBS: 0.2350012477048381
Fold 5 IBS: 0.22812847237666628
[I 2024-04-16 23:05:26,584] Trial 90 finished with value: 0.2222431751809597 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 7, 'min_samples_leaf': 9, 'max_depth': 14, 'n_estimators': 427, 'oob_score': True, 'warm_start': False, 'max_features': 0.1, 'max_samples': 0.9384169100812306, 'min_weight_fraction_leaf': 0.10055258156251395}. Best is trial 82 with value: 0.20797277264373898.
Fold 1 IBS: 0.23494236000327381
Fold 2 IBS: 0.19851393414446025
Fold 3 IBS: 0.188316054669442
Fold 4 IBS: 0.20779679335240808
Fold 5 IBS: 0.21280969619087245
[I 2024-04-16 23:05:31,830] Trial 91 finished with value: 0.20847576767209133 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 8, 'min_samples_leaf': 10, 'max_depth': 12, 'n_estimators': 442, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.961995

In [91]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [92]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.842
train_ibs:  0.208


#### Test

In [93]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [94]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=16, max_features=None, max_leaf_nodes=10,
                   max_samples=0.9186714469202719, min_samples_leaf=2,
                   min_samples_split=3,
                   min_weight_fraction_leaf=0.04956115599708024,
                   n_estimators=474, random_state=123, warm_start=True)

C-index score: 0.567


ExtraSurvivalTrees(max_depth=12, max_features=None, max_leaf_nodes=9,
                   max_samples=0.9407754324156289, min_samples_leaf=9,
                   min_samples_split=2,
                   min_weight_fraction_leaf=0.04206241234084829,
                   n_estimators=431, oob_score=True, random_state=123)

IBS: 0.235


In [95]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [96]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-16 23:06:15,921] A new study created in memory with name: no-name-2745463c-8dd7-400a-8b0a-4145aad16972


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 23:06:34,487] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 23:06:42,481] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 23:12:06,350] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 12 with value: 0.6841431930275611.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 23:12:44,425] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'squar

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 23:19:43,765] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.8502968404151126, 'learning_rate': 0.024443951259730985, 'dropout_rate': 0.2675273969344081, 'n_estimators': 441, 'criterion': 'squared_error', 'ccp_alpha': 2.0753749717266823, 'min_weight_fraction_leaf': 0.3503497788125578, 'max_features': 'auto', 'min_impurity_decrease': 7.237153572123947e-07, 'validation_fraction': 0.41222573804914475, 'min_samples_split': 16, 'max_leaf_nodes': 13, 'min_samples_leaf': 12, 'max_depth': 3}. Best is trial 12 with value: 0.6841431930275611.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 23:19:56,298] Trial 26 finished with value: 0.5 and parameters: {'subsample': 0.7670085127536703, 'learning_rate': 0.011828778593594373, 'dropout_rate': 0.4300954216773497, 'n_estimators': 330, 'criterion': 'friedm

...

Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.22939559304809248
[I 2024-04-17 00:00:10,614] Trial 55 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.809323041898869, 'learning_rate': 0.0040826970179266685, 'dropout_rate': 0.36065535241543395, 'n_estimators': 435, 'criterion': 'squared_error', 'ccp_alpha': 1.5126136571072866, 'min_weight_fraction_leaf': 0.2002957617254777, 'max_features': 'auto', 'min_impurity_decrease': 2.0868015436948727e-05, 'validation_fraction': 0.8814649292460881, 'min_samples_split': 17, 'max_leaf_nodes': 18, 'min_samples_leaf': 11, 'max_depth': 2}. Best is trial 41 with value: 0.23455338278002208.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939559304809248
[I 2024-04-17 00:00:29,934] Trial 56 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.2818850250060197, 'learning_rate': 0.0985092

Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.2293955930480925
[I 2024-04-17 00:04:33,454] Trial 66 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.8489175059797514, 'learning_rate': 0.008448226967076113, 'dropout_rate': 0.12725750793823018, 'n_estimators': 424, 'criterion': 'squared_error', 'ccp_alpha': 0.893877768639663, 'min_weight_fraction_leaf': 0.2702972473075093, 'max_features': 'auto', 'min_impurity_decrease': 7.586810829424823e-05, 'validation_fraction': 0.8971088342813041, 'min_samples_split': 17, 'max_leaf_nodes': 17, 'min_samples_leaf': 8, 'max_depth': 4}. Best is trial 62 with value: 0.23436365650963703.
Fold 1 IBS: 0.24724710044658996
Fold 2 IBS: 0.23203988453792296
Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-17 00:04:54,471] Trial 67 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9594402941587806,

Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939559304809248
[I 2024-04-17 00:08:36,895] Trial 77 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.7234820637720293, 'learning_rate': 0.04128077713454106, 'dropout_rate': 0.19884799526452568, 'n_estimators': 302, 'criterion': 'squared_error', 'ccp_alpha': 1.2832467794713243, 'min_weight_fraction_leaf': 0.4027406894595495, 'max_features': None, 'min_impurity_decrease': 5.35591590419612e-07, 'validation_fraction': 0.8758106078120852, 'min_samples_split': 19, 'max_leaf_nodes': 5, 'min_samples_leaf': 10, 'max_depth': 2}. Best is trial 70 with value: 0.23410964446224067.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-17 00:08:52,972] Trial 78 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.6708593402641205, '

Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.22939559304809248
[I 2024-04-17 00:11:40,060] Trial 88 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.6879348852579991, 'learning_rate': 0.08209035744510738, 'dropout_rate': 0.24522201083374043, 'n_estimators': 84, 'criterion': 'squared_error', 'ccp_alpha': 0.2983993026610602, 'min_weight_fraction_leaf': 0.3774672254272644, 'max_features': None, 'min_impurity_decrease': 5.4011226362110275e-06, 'validation_fraction': 0.568843188657281, 'min_samples_split': 16, 'max_leaf_nodes': 6, 'min_samples_leaf': 4, 'max_depth': 11}. Best is trial 81 with value: 0.23062671367871226.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977708
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-17 00:11:42,875] Trial 89 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.6555693108457097, 'learning_rate': 0.08348043724232

Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-17 00:13:34,563] Trial 99 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.4750010457704055, 'learning_rate': 0.07813501740629669, 'dropout_rate': 0.30318261654818046, 'n_estimators': 242, 'criterion': 'squared_error', 'ccp_alpha': 1.7079733343503498, 'min_weight_fraction_leaf': 0.32269779176244495, 'max_features': 0.1, 'min_impurity_decrease': 1.276230329676773e-06, 'validation_fraction': 0.36954438153697, 'min_samples_split': 11, 'max_leaf_nodes': 9, 'min_samples_leaf': 7, 'max_depth': 2}. Best is trial 91 with value: 0.2300644165494142.


* Best trial for IBS: 
 FrozenTrial(number=91, state=TrialState.COMPLETE, values=[0.2300644165494142], datetime_start=datetime.datetime(2024, 4, 17, 0, 11, 48, 159403), datetime_complete=datetime.datetime(2024, 4, 17, 0, 12, 5, 711292), params={'subsample': 0.579213280645811, 'learning_rate': 0.09426091382038485, 'dropout_rate': 0.18479264516541616, 'n

In [97]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [98]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.7
train_ibs:  0.23


#### Test

In [99]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [100]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.004507507039685693,
                                 criterion='squared_error',
                                 dropout_rate=0.8022681311855436,
                                 learning_rate=0.010888929954369636,
                                 max_depth=12, max_features='auto',
                                 max_leaf_nodes=3,
                                 min_impurity_decrease=4.0700060224631717e-07,
                                 min_samples_leaf=10, min_samples_split=19,
                                 min_weight_fraction_leaf=0.1685851039915477,
                                 n_estimators=443, random_state=123,
                                 subsample=0.8597838031871279,
                                 validation_fraction=0.8827408742810589)

C-index score: 0.516


GradientBoostingSurvivalAnalysis(ccp_alpha=0.03787089343697578,
                                 criterion='squared_error',
                                 dropout_rate=0.18479264516541616,
                                 learning_rate=0.09426091382038485, max_depth=2,
                                 max_leaf_nodes=6,
                                 min_impurity_decrease=7.4293259533530295e-06,
                                 min_samples_leaf=7, min_samples_split=18,
                                 min_weight_fraction_leaf=0.4036747584607865,
                                 n_estimators=390, random_state=123,
                                 subsample=0.579213280645811,
                                 validation_fraction=0.5325834925366492)

IBS: 0.228


In [101]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [102]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [103]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-17 00:13:42,362] A new study created in memory with name: no-name-254a1012-5239-41ed-83ef-cdb877a6703b


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6374501992031872
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6539923954372624
Fold 5 C-index: 0.6609442060085837
[I 2024-04-17 00:13:46,934] Trial 0 finished with value: 0.6423526693826518 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.6423526693826518.
Fold 1 C-index: 0.6653386454183267
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6425855513307985
Fold 5 C-index: 0.6609442060085837
[I 2024-04-17 00:13:59,998] Trial 1 finished with value: 0.6456489898043869 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 1 with value: 0.6456489898043869.
Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 

Fold 1 C-index: 0.6095617529880478
Fold 2 C-index: 0.7015503875968992
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.688212927756654
Fold 5 C-index: 0.6609442060085837
[I 2024-04-17 00:16:16,448] Trial 19 finished with value: 0.6443942804019518 and parameters: {'subsample': 0.38793667853472646, 'dropout_rate': 0.7077875554849441, 'n_estimators': 85, 'learning_rate': 0.04269105711861253}. Best is trial 7 with value: 0.6499035318694023.
Fold 1 C-index: 0.6613545816733067
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6539923954372624
Fold 5 C-index: 0.6609442060085837
[I 2024-04-17 00:16:25,778] Trial 20 finished with value: 0.6471335458766757 and parameters: {'subsample': 0.6004927729383457, 'dropout_rate': 0.608992508329216, 'n_estimators': 310, 'learning_rate': 0.02073176682253709}. Best is trial 7 with value: 0.6499035318694023.
Fold 1 C-index: 0.6613545816733067
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 

Fold 1 C-index: 0.6613545816733067
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6387832699619772
Fold 5 C-index: 0.6609442060085837
[I 2024-04-17 00:18:11,849] Trial 38 finished with value: 0.6440917207816187 and parameters: {'subsample': 0.7910265829012886, 'dropout_rate': 0.8201500943507289, 'n_estimators': 1, 'learning_rate': 0.0802388198308747}. Best is trial 7 with value: 0.6499035318694023.
Fold 1 C-index: 0.6613545816733067
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4 C-index: 0.6539923954372624
Fold 5 C-index: 0.6609442060085837
[I 2024-04-17 00:18:17,538] Trial 39 finished with value: 0.6471335458766757 and parameters: {'subsample': 0.6483381968153921, 'dropout_rate': 0.6826761322925258, 'n_estimators': 159, 'learning_rate': 0.04656913718567279}. Best is trial 7 with value: 0.6499035318694023.
Fold 1 C-index: 0.6653386454183267
Fold 2 C-index: 0.6976744186046512
Fold 3 C-index: 0.5617021276595745
Fold 4

...

Fold 3 IBS: 0.27412770407771303
Fold 4 IBS: 0.2250727043053573
Fold 5 IBS: 0.2280063139706162
[I 2024-04-17 00:23:34,631] Trial 19 finished with value: 0.23082306581647552 and parameters: {'subsample': 0.3892284838807411, 'dropout_rate': 0.5354432466556798, 'n_estimators': 131, 'learning_rate': 0.023294697221931306}. Best is trial 7 with value: 0.22271699753958316.
Fold 1 IBS: 0.22813963029627435
Fold 2 IBS: 0.21261919492535192
Fold 3 IBS: 0.2328032377684259
Fold 4 IBS: 0.23177471930860935
Fold 5 IBS: 0.21241390693232276
[I 2024-04-17 00:23:38,158] Trial 20 finished with value: 0.22355013784619687 and parameters: {'subsample': 0.6210873354088753, 'dropout_rate': 0.7933084651006226, 'n_estimators': 66, 'learning_rate': 0.013007963411749002}. Best is trial 7 with value: 0.22271699753958316.
Fold 1 IBS: 0.23294882482388735
Fold 2 IBS: 0.2172731528360657
Fold 3 IBS: 0.23037554967791973
Fold 4 IBS: 0.23393962524579212
Fold 5 IBS: 0.21554329357937044
[I 2024-04-17 00:23:41,848] Trial 21 fini

Fold 4 IBS: 0.22761235729593224
Fold 5 IBS: 0.21391136761511492
[I 2024-04-17 00:24:49,806] Trial 38 finished with value: 0.22439947178405434 and parameters: {'subsample': 0.9323458933659868, 'dropout_rate': 0.13272164755980653, 'n_estimators': 28, 'learning_rate': 0.0802388198308747}. Best is trial 30 with value: 0.22105020743663645.
Fold 1 IBS: 0.22935008117918856
Fold 2 IBS: 0.21153618549256686
Fold 3 IBS: 0.29072451950301204
Fold 4 IBS: 0.2586656669929489
Fold 5 IBS: 0.24321840787686647
[I 2024-04-17 00:24:53,662] Trial 39 finished with value: 0.24669897220891657 and parameters: {'subsample': 0.7076071631229807, 'dropout_rate': 0.36113352605756727, 'n_estimators': 75, 'learning_rate': 0.05598015324329953}. Best is trial 30 with value: 0.22105020743663645.
Fold 1 IBS: 0.4295827306814518
Fold 2 IBS: 0.33729766730917193
Fold 3 IBS: 0.38924638966253877
Fold 4 IBS: 0.31325291352447
Fold 5 IBS: 0.32175848063875884
[I 2024-04-17 00:25:05,734] Trial 40 finished with value: 0.35822763636327

Fold 5 IBS: 0.28019979914084864
[I 2024-04-17 00:26:15,910] Trial 57 finished with value: 0.28649058779241166 and parameters: {'subsample': 0.6825327446543555, 'dropout_rate': 0.5208726510676236, 'n_estimators': 282, 'learning_rate': 0.025565265724993874}. Best is trial 55 with value: 0.22089716088725195.
Fold 1 IBS: 0.2214507245251379
Fold 2 IBS: 0.20506568938296435
Fold 3 IBS: 0.24075353489551313
Fold 4 IBS: 0.2288359521484131
Fold 5 IBS: 0.2104012829571286
[I 2024-04-17 00:26:19,034] Trial 58 finished with value: 0.22130143678183142 and parameters: {'subsample': 0.5793765216559574, 'dropout_rate': 0.869359581862351, 'n_estimators': 38, 'learning_rate': 0.03754893539001649}. Best is trial 55 with value: 0.22089716088725195.
Fold 1 IBS: 0.22332388143407295
Fold 2 IBS: 0.20363163283585628
Fold 3 IBS: 0.24449748198927226
Fold 4 IBS: 0.22907918428885007
Fold 5 IBS: 0.21077017722074096
[I 2024-04-17 00:26:22,292] Trial 59 finished with value: 0.22226047155375853 and parameters: {'subsampl

Fold 1 IBS: 0.22462298117263793
Fold 2 IBS: 0.2073982536472491
Fold 3 IBS: 0.28361457332876805
Fold 4 IBS: 0.24675247717794
Fold 5 IBS: 0.2381675349082399
[I 2024-04-17 00:27:32,298] Trial 77 finished with value: 0.240111164046967 and parameters: {'subsample': 0.541487090871485, 'dropout_rate': 0.7704979082473631, 'n_estimators': 165, 'learning_rate': 0.02189762732192369}. Best is trial 55 with value: 0.22089716088725195.
Fold 1 IBS: 0.22365763185049595
Fold 2 IBS: 0.20591143440013582
Fold 3 IBS: 0.280976248290565
Fold 4 IBS: 0.24848562445786387
Fold 5 IBS: 0.23615081501710733
[I 2024-04-17 00:27:37,119] Trial 78 finished with value: 0.2390363508032336 and parameters: {'subsample': 0.6733814576309736, 'dropout_rate': 0.7358591929789065, 'n_estimators': 128, 'learning_rate': 0.02725879214599771}. Best is trial 55 with value: 0.22089716088725195.
Fold 1 IBS: 0.22811684389438022
Fold 2 IBS: 0.21004117151133037
Fold 3 IBS: 0.23552715888942047
Fold 4 IBS: 0.22642754442179203
Fold 5 IBS: 0.2

Fold 2 IBS: 0.20078159037918683
Fold 3 IBS: 0.26110289322293423
Fold 4 IBS: 0.23412808505560326
Fold 5 IBS: 0.21973897733190245
[I 2024-04-17 00:28:45,871] Trial 96 finished with value: 0.22673056855702786 and parameters: {'subsample': 0.7054650333977128, 'dropout_rate': 0.6262161752789943, 'n_estimators': 61, 'learning_rate': 0.03998698220337849}. Best is trial 81 with value: 0.2208903160925389.
Fold 1 IBS: 0.21963880593620028
Fold 2 IBS: 0.20092268436766023
Fold 3 IBS: 0.2648475542088438
Fold 4 IBS: 0.2331055121729352
Fold 5 IBS: 0.22220093036998428
[I 2024-04-17 00:28:49,773] Trial 97 finished with value: 0.22814309741112476 and parameters: {'subsample': 0.531829760385333, 'dropout_rate': 0.6847533776104763, 'n_estimators': 79, 'learning_rate': 0.03223905987322316}. Best is trial 81 with value: 0.2208903160925389.
Fold 1 IBS: 0.2466414230044532
Fold 2 IBS: 0.23146700010719856
Fold 3 IBS: 0.2289119634694628
Fold 4 IBS: 0.24164291053866815
Fold 5 IBS: 0.2288108925224838
[I 2024-04-17 

In [104]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [105]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.67
train_ibs:  0.221


#### Test

In [106]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [107]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.9036811565813793,
                                              learning_rate=0.03163367036018912,
                                              n_estimators=1, random_state=123,
                                              subsample=0.2748398992643779)

C-index score: 0.521


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.8767395889398294,
                                              learning_rate=0.030725485144314495,
                                              n_estimators=48, random_state=123,
                                              subsample=0.5967091670188371)

IBS: 0.239


In [108]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [109]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.847,1.0
ExtraSurvivalTrees,0.842,2.0
GradientBoosting,0.700,3.0
ComponentwiseGradientBoosting,0.670,4.0
CoxElastic,0.641,5.0
CoxRidge,0.635,6.0
CoxLasso,0.583,7.0


In [110]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
ExtraSurvivalTrees,0.208,1.0
Randomsurvivalforest,0.212,2.0
ComponentwiseGradientBoosting,0.221,3.0
GradientBoosting,0.230,4.0
CoxElastic,0.234,5.0
CoxRidge,0.236,6.0
CoxLasso,0.353,7.0


In [111]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
ExtraSurvivalTrees,0.567,1.0
Randomsurvivalforest,0.540,2.0
CoxRidge,0.530,3.0
CoxElastic,0.527,4.0
ComponentwiseGradientBoosting,0.521,5.0
GradientBoosting,0.516,6.0
CoxLasso,0.491,7.0


In [112]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs

,IBS,rank
GradientBoosting,0.228,1.0
CoxRidge,0.229,2.5
CoxElastic,0.229,2.5
ExtraSurvivalTrees,0.235,4.0
ComponentwiseGradientBoosting,0.239,5.0
Randomsurvivalforest,0.262,6.0
CoxLasso,0.457,7.0


In [113]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = 'path_to_your_folder/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d3/dfs/yeojohnson/no_selection/'  # Folder path where you want to save the files

# List of corresponding file namess
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d3_dfs_yeojohnson_no_selection_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [114]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-17
